# Kling V3 Image-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Kling V3 (Image-to-Video)** 模型 API。

## 模型简介

Kling V3 是快手推出的最新一代视频生成模型，支持以下能力：

- **起始帧动画**：将静态图片作为起始帧，根据提示词生成动态视频
- **首尾帧过渡**：支持同时指定起始帧和结束帧，生成平滑的过渡视频
- **Native 4K**：支持原生 4K 分辨率生成
- **音频生成**：支持生成同步音频
- **高级控制**：支持镜头控制、多镜头剪辑等高级功能

**API 端点**：`fal-ai/kling-video/v3/{mode}/image-to-video`

**Mode 选项**：`standard`（720P）、`pro`（1080P）、`4k`（Native 4K）

**注意**：V3 使用 `start_image_url` 和 `end_image_url` 参数

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 起始帧生成视频

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：起始帧 + 提示词
result = run_fal_task(
    "fal-ai/kling-video/v3/standard/image-to-video",
    arguments={
        "start_image_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-v3/pro-i2v/start_image.png"
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 描述视频动作和运动的提示词 |
| `start_image_url` | string | *必填* | 起始帧图片 URL（支持 JPEG/PNG/WebP） |
| `end_image_url` | string | 可选 | 结束帧图片 URL，创建首尾帧之间的过渡效果 |
| `negative_prompt` | string | 可选 | 负面提示词 |
| `duration` | string | `"5"` | 视频时长：`"5"` 或 `"10"` 秒 |
| `cfg_scale` | float | 可选 | 生成强度，控制与提示词的匹配程度 |
| `generate_audio` | boolean | `true` | 是否生成与视频同步的音频 |
| `shot_type` | string | 可选 | 镜头类型，使用 `multi_prompt` 时必填 |
| `multi_prompt` | list | 可选 | 多镜头提示词列表 |
| `elements` | list | 可选 | 元素列表（风格、主体等参考） |

## 3. 首尾帧过渡示例

通过同时指定 `start_image_url`（起始帧）和 `end_image_url`（结束帧），模型会生成两帧之间的平滑过渡视频。

In [ ]:
# 首尾帧过渡示例
result_transition = run_fal_task(
    "fal-ai/kling-video/v3/pro/image-to-video",
    arguments={
        "prompt": "从白天平滑过渡到夜晚，城市灯光逐渐亮起",
        "start_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
        "end_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
        "duration": "10",
    },
)

# 展示结果
video_url = result_transition["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 4. Native 4K 示例

Kling V3 支持原生 4K 分辨率生成，提供极致的画质体验。

In [ ]:
# Native 4K 示例
result_4k = run_fal_task(
    "fal-ai/kling-video/v3/4k/image-to-video",
    arguments={
        "prompt": "画面中的人物开始跑步，背景快速后退",
        "start_image_url": "https://aitoken-public.qnaigc.com/example/generate-video/running-man.jpg",
        "duration": "10",
    },
)

# 展示结果
video_url = result_4k["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. 高级用法

使用更多可选参数来精细控制视频生成效果。

In [ ]:
# 高级用法：使用更多参数
result_advanced = run_fal_task(
    "fal-ai/kling-video/v3/pro/image-to-video",
    arguments={
        "prompt": "The craftsman slowly examines the bowl, turning it gently in his weathered hands. His eyes reflect years of wisdom. Subtle smile forms on his face. Dust particles drift in warm light. Breathing motion, blinking eyes.",
        "multi_prompt": None,
        "start_image_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-v3/pro-i2v/start_image.png",
        "duration": "10",
        "generate_audio": True,
        "elements": [
            {
            "reference_image_urls": [
                "https://v3b.fal.media/files/b/0a8cfd62/psPCmzrD1y9vDgdyNfKAL_glasses_back.png"
            ],
            "frontal_image_url": "https://v3b.fal.media/files/b/0a8cfd5f/-kZL-ha3Iuelku5IHXC-A_glasses.png"
            },
            {
            "video_url": "https://v3b.fal.media/files/b/0a8cfd66/b03SOiQvKLlFx_jqdNZ9z_child_video.mp4"
            }
        ],
        "shot_type": "customize",
        "negative_prompt": "blur, distort, and low quality",
        "cfg_scale": 0.5
    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "fal-ai/kling-video/v3/standard/image-to-video",
    arguments={
         "start_image_url": "https://storage.googleapis.com/falserverless/example_inputs/kling-v3/pro-i2v/start_image.png"
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "fal-ai/kling-video/v3/standard/image-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
        "fal-ai/kling-video/v3/standard/image-to-video",
        request_id,
    )

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 7. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填)",
  "start_image_url": "string (必填, 起始帧图片 URL)",
  "end_image_url": "string (可选, 结束帧图片 URL)",
  "negative_prompt": "string (可选)",
  "duration": "5 | 10 (可选, 默认 5)",
  "cfg_scale": "float (可选)",
  "generate_audio": "boolean (可选, 默认 true)",
  "shot_type": "string (可选, 使用 multi_prompt 时必填)",
  "multi_prompt": ["object (可选, 多镜头提示词列表)"],
  "elements": ["object (可选, 元素列表)"]
}
```

### Output Schema

```json
{
  "video": {
    "file_name": "string",
    "url": "string",
    "content_type": "string",
    "file_size": "number"
  }
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Kling V3 模型页面](https://fal.ai/models/fal-ai/kling-video/v3/standard/image-to-video)